In [1]:
import sys
import os
from google.colab import drive

# 1. Monta o Google Drive dentro da VM
# Vai pedir permissão/login na primeira vez
drive.mount('/content/drive')

# 2. Define o caminho onde você salvou a pasta no Drive
# Ajuste o caminho abaixo se você salvou dentro de subpastas
project_path = '/content/drive/MyDrive/u3'

# 3. Adiciona ao Path do Python
if project_path not in sys.path:
    sys.path.append(project_path)

# 4. Teste de verificação
print(f"Tentando acessar: {project_path}")
if os.path.exists(os.path.join(project_path, 'src')):
    print("SUCESSO: Pasta 'src' encontrada no Google Drive!")
else:
    print("ERRO: Pasta não encontrada. Verifique se o nome no Drive está igual.")

# --- SEUS IMPORTS ---
import torch
from src.dataset import get_svhn_loaders
from src.model import SVHNNet
from src.train_utils import capture_optimizer_internals
from src.visualization import plot_gradients_distribution

ValueError: mount failed

In [1]:
import sys
import os
# Adiciona a pasta src ao path para importar os módulos
sys.path.append(os.path.abspath(os.path.join('..')))

import torch
import torch.nn as nn
import torch.optim as optim
from src.dataset import get_svhn_loaders
from src.model import SVHNNet
from src.train_utils import capture_optimizer_internals
from src.visualization import plot_gradients_distribution

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando device: {device}")

ModuleNotFoundError: No module named 'src'

## Loop de Treino Customizado para Captura

In [ ]:
# Carrega dados
train_loader, test_loader = get_svhn_loaders(batch_size=64)

# Inicializa Modelo e Otimizador (Adam é obrigatório para ver a adaptação)
model = SVHNNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Listas para guardar snapshots dos gradientes
grads_history = []

# Treina por apenas 1 época para demonstração (ou mais se quiser)
model.train()
print("Iniciando treino para captura de gradientes...")

for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)
    
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    
    # --- CAPTURA ANTES DO STEP ---
    # Capturamos a cada 100 batches para não lotar a memória
    if batch_idx % 100 == 0:
        raw, ewma, adapted = capture_optimizer_internals(model, optimizer, layer_name='conv1')
        if raw is not None:
            grads_history.append((batch_idx, raw, ewma, adapted))
    
    optimizer.step()

print("Captura finalizada.")

## Visualização (Gera os gráficos do Tópico 3)

In [ ]:
# Pega o último snapshot capturado
idx, raw, ewma, adapted = grads_history[-1]

print(f"Visualizando distribuição do Batch {idx}")
plot_gradients_distribution(raw, ewma, adapted, epoch_idx=1)